# 04. 실험 결과 분석 및 비교

이 노트북에서는 3개 실험(exp-001, exp-002, exp-003)의 결과를 종합 분석하고 비교합니다.

## 목차
1. 환경 설정
2. 실험 결과 로드
3. 학습 곡선 비교
4. Tool Calling 정확도 비교
5. 종합 분석 및 인사이트
6. 최종 보고서 생성

## 1. 환경 설정

In [ ]:
import os
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import yaml

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# 색상 팔레트
COLORS = {
    'exp-001': '#1f77b4',  # Blue
    'exp-002': '#ff7f0e',  # Orange
    'exp-003': '#2ca02c',  # Green
}

print("환경 설정 완료!")

In [ ]:
# 실험 목록 및 설정 정의
EXPERIMENTS = ['exp-001', 'exp-002', 'exp-003']

# 각 실험의 설정 로드
configs = {}
for exp in EXPERIMENTS:
    config_path = f'../configs/training_config_{exp.replace("-", "")}.yaml'
    if os.path.exists(config_path):
        with open(config_path, 'r', encoding='utf-8') as f:
            configs[exp] = yaml.safe_load(f)
        print(f"✓ {exp}: {configs[exp]['experiment']['description']}")
    else:
        print(f"✗ {exp}: 설정 파일 없음")

## 2. 실험 결과 로드

In [ ]:
# 학습 로그 로드
training_logs = {}

for exp in EXPERIMENTS:
    if exp not in configs:
        continue
    
    log_path = f"../{configs[exp]['misc']['output_dir']}/training_log.json"
    
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            training_logs[exp] = json.load(f)
        print(f"✓ {exp}: 학습 로그 로드 완료 ({len(training_logs[exp])} entries)")
    else:
        print(f"✗ {exp}: 학습 로그 없음 - 02_qlora_training.ipynb 실행 필요")

In [ ]:
# 평가 결과 로드
evaluation_results = {}

for exp in EXPERIMENTS:
    if exp not in configs:
        continue
    
    eval_path = f"../{configs[exp]['misc']['output_dir']}/evaluation_summary.json"
    
    if os.path.exists(eval_path):
        with open(eval_path, 'r', encoding='utf-8') as f:
            evaluation_results[exp] = json.load(f)
        print(f"✓ {exp}: 평가 결과 로드 완료 (정확도: {evaluation_results[exp]['overall_accuracy']:.2f}%)")
    else:
        print(f"✗ {exp}: 평가 결과 없음 - 03_evaluation.ipynb 실행 필요")

## 3. 학습 곡선 비교

In [ ]:
# 학습 데이터가 있는 실험만 처리
available_exps = [exp for exp in EXPERIMENTS if exp in training_logs]

if not available_exps:
    print("아직 학습된 실험이 없습니다. 02_qlora_training.ipynb를 먼저 실행하세요.")
else:
    print(f"분석 가능한 실험: {available_exps}")

In [ ]:
# Training Loss 비교
if available_exps:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Training Loss
    for exp in available_exps:
        train_loss = [(x['step'], x['loss']) 
                      for x in training_logs[exp] 
                      if 'loss' in x and 'eval_loss' not in x]
        if train_loss:
            steps, losses = zip(*train_loss)
            axes[0].plot(steps, losses, label=exp, color=COLORS[exp], linewidth=2)
    
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Validation Loss
    for exp in available_exps:
        eval_loss = [(x['step'], x['eval_loss']) 
                     for x in training_logs[exp] 
                     if 'eval_loss' in x]
        if eval_loss:
            steps, losses = zip(*eval_loss)
            axes[1].plot(steps, losses, label=exp, color=COLORS[exp], linewidth=2, marker='o')
    
    axes[1].set_xlabel('Steps')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss Comparison')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/loss_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Loss 비교 그래프 저장: results/loss_comparison.png")

In [ ]:
# 최종 Loss 값 비교
if available_exps:
    loss_summary = []
    
    for exp in available_exps:
        train_loss = [x['loss'] for x in training_logs[exp] if 'loss' in x and 'eval_loss' not in x]
        eval_loss = [x['eval_loss'] for x in training_logs[exp] if 'eval_loss' in x]
        
        loss_summary.append({
            'Experiment': exp,
            'Description': configs[exp]['experiment']['description'],
            'Final Train Loss': train_loss[-1] if train_loss else None,
            'Final Eval Loss': eval_loss[-1] if eval_loss else None,
            'Min Eval Loss': min(eval_loss) if eval_loss else None,
        })
    
    df_loss = pd.DataFrame(loss_summary)
    print("\n최종 Loss 비교:")
    print(df_loss.to_string(index=False))

## 4. Tool Calling 정확도 비교

In [ ]:
# 평가 결과가 있는 실험만 처리
eval_available_exps = [exp for exp in EXPERIMENTS if exp in evaluation_results]

if not eval_available_exps:
    print("아직 평가된 실험이 없습니다. 03_evaluation.ipynb를 먼저 실행하세요.")
else:
    print(f"평가 결과가 있는 실험: {eval_available_exps}")

In [ ]:
# 전체 정확도 비교
if eval_available_exps:
    accuracy_data = []
    
    for exp in eval_available_exps:
        result = evaluation_results[exp]
        accuracy_data.append({
            'Experiment': exp,
            'Description': configs[exp]['experiment']['description'],
            'Accuracy (%)': result['overall_accuracy'],
            'Correct': result['correct_count'],
            'Total': result['total_test_cases']
        })
    
    df_accuracy = pd.DataFrame(accuracy_data)
    print("\n전체 정확도 비교:")
    print(df_accuracy.to_string(index=False))

In [ ]:
# Tool별 정확도 비교 시각화
if eval_available_exps:
    # Tool 목록 추출
    all_tools = set()
    for exp in eval_available_exps:
        all_tools.update(evaluation_results[exp]['accuracy_by_tool'].keys())
    all_tools = sorted(all_tools)
    
    # 데이터 준비
    tool_accuracy_data = []
    for exp in eval_available_exps:
        for tool in all_tools:
            if tool in evaluation_results[exp]['accuracy_by_tool']:
                acc = evaluation_results[exp]['accuracy_by_tool'][tool]['accuracy']
            else:
                acc = 0
            tool_accuracy_data.append({
                'Experiment': exp,
                'Tool': tool,
                'Accuracy': acc
            })
    
    df_tool_acc = pd.DataFrame(tool_accuracy_data)
    
    # 그룹 바 차트
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x = np.arange(len(all_tools))
    width = 0.25
    
    for i, exp in enumerate(eval_available_exps):
        exp_data = df_tool_acc[df_tool_acc['Experiment'] == exp]
        accuracies = [exp_data[exp_data['Tool'] == t]['Accuracy'].values[0] for t in all_tools]
        ax.bar(x + i * width, accuracies, width, label=exp, color=COLORS[exp])
    
    ax.set_xlabel('Tool')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Tool-wise Accuracy Comparison')
    ax.set_xticks(x + width)
    ax.set_xticklabels(all_tools, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../results/tool_accuracy_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Tool별 정확도 비교 그래프 저장: results/tool_accuracy_comparison.png")

In [ ]:
# 복잡도별 정확도 비교
if eval_available_exps:
    # 복잡도 목록 추출
    all_complexities = set()
    for exp in eval_available_exps:
        all_complexities.update(evaluation_results[exp]['accuracy_by_complexity'].keys())
    all_complexities = sorted(all_complexities)
    
    # 데이터 준비
    complexity_accuracy_data = []
    for exp in eval_available_exps:
        for comp in all_complexities:
            if comp in evaluation_results[exp]['accuracy_by_complexity']:
                acc = evaluation_results[exp]['accuracy_by_complexity'][comp]['accuracy']
            else:
                acc = 0
            complexity_accuracy_data.append({
                'Experiment': exp,
                'Complexity': comp,
                'Accuracy': acc
            })
    
    df_comp_acc = pd.DataFrame(complexity_accuracy_data)
    
    # Heatmap
    pivot_df = df_comp_acc.pivot(index='Experiment', columns='Complexity', values='Accuracy')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='RdYlGn', 
                vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Accuracy (%)'})
    ax.set_title('Accuracy by Experiment and Complexity')
    
    plt.tight_layout()
    plt.savefig('../results/complexity_accuracy_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("복잡도별 정확도 히트맵 저장: results/complexity_accuracy_heatmap.png")

In [ ]:
# 에러 유형 분포 비교
if eval_available_exps:
    fig, axes = plt.subplots(1, len(eval_available_exps), figsize=(6*len(eval_available_exps), 5))
    
    if len(eval_available_exps) == 1:
        axes = [axes]
    
    for i, exp in enumerate(eval_available_exps):
        error_dist = evaluation_results[exp].get('error_distribution', {})
        
        if error_dist:
            axes[i].pie(
                error_dist.values(),
                labels=error_dist.keys(),
                autopct='%1.1f%%',
                startangle=90
            )
        else:
            axes[i].text(0.5, 0.5, 'No Errors!', ha='center', va='center', fontsize=16)
        
        axes[i].set_title(f'{exp}\nError Distribution')
    
    plt.tight_layout()
    plt.savefig('../results/error_distribution_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("에러 분포 비교 그래프 저장: results/error_distribution_comparison.png")

## 5. 종합 분석 및 인사이트

In [ ]:
# 실험 설정 비교 테이블
config_comparison = []

for exp in EXPERIMENTS:
    if exp not in configs:
        continue
    
    cfg = configs[exp]
    config_comparison.append({
        'Experiment': exp,
        'Learning Rate': cfg['training']['learning_rate'],
        'Batch Size': cfg['training']['per_device_train_batch_size'],
        'Epochs': cfg['training']['num_train_epochs'],
        'LoRA Rank': cfg['lora']['r'],
        'LoRA Alpha': cfg['lora']['lora_alpha'],
    })

df_config = pd.DataFrame(config_comparison)
print("\n실험 설정 비교:")
print(df_config.to_string(index=False))

In [ ]:
# 종합 결과 테이블
if eval_available_exps:
    comprehensive_results = []
    
    for exp in eval_available_exps:
        cfg = configs[exp]
        result = evaluation_results[exp]
        
        # 학습 로그에서 최종 loss 추출
        final_train_loss = None
        final_eval_loss = None
        if exp in training_logs:
            train_losses = [x['loss'] for x in training_logs[exp] if 'loss' in x and 'eval_loss' not in x]
            eval_losses = [x['eval_loss'] for x in training_logs[exp] if 'eval_loss' in x]
            if train_losses:
                final_train_loss = train_losses[-1]
            if eval_losses:
                final_eval_loss = eval_losses[-1]
        
        comprehensive_results.append({
            'Experiment': exp,
            'LR': cfg['training']['learning_rate'],
            'Rank': cfg['lora']['r'],
            'Epochs': cfg['training']['num_train_epochs'],
            'Train Loss': f"{final_train_loss:.4f}" if final_train_loss else 'N/A',
            'Eval Loss': f"{final_eval_loss:.4f}" if final_eval_loss else 'N/A',
            'Accuracy (%)': f"{result['overall_accuracy']:.2f}",
        })
    
    df_comprehensive = pd.DataFrame(comprehensive_results)
    print("\n종합 결과 비교:")
    print(df_comprehensive.to_string(index=False))

In [ ]:
# 최고 성능 실험 찾기
if eval_available_exps:
    best_exp = max(eval_available_exps, 
                   key=lambda x: evaluation_results[x]['overall_accuracy'])
    best_accuracy = evaluation_results[best_exp]['overall_accuracy']
    
    print("\n" + "=" * 60)
    print("최고 성능 실험")
    print("=" * 60)
    print(f"\n실험: {best_exp}")
    print(f"설명: {configs[best_exp]['experiment']['description']}")
    print(f"정확도: {best_accuracy:.2f}%")
    print(f"\n주요 설정:")
    print(f"  - Learning Rate: {configs[best_exp]['training']['learning_rate']}")
    print(f"  - LoRA Rank: {configs[best_exp]['lora']['r']}")
    print(f"  - Epochs: {configs[best_exp]['training']['num_train_epochs']}")

In [ ]:
# 인사이트 생성
insights = []

if len(eval_available_exps) >= 2:
    # 정확도 비교
    accuracies = {exp: evaluation_results[exp]['overall_accuracy'] for exp in eval_available_exps}
    
    # LR 영향 분석 (exp-001 vs exp-002)
    if 'exp-001' in accuracies and 'exp-002' in accuracies:
        lr_diff = accuracies['exp-002'] - accuracies['exp-001']
        if lr_diff > 0:
            insights.append(f"Learning Rate 감소(2e-4 → 1e-4)로 정확도가 {lr_diff:.2f}% 향상되었습니다.")
        else:
            insights.append(f"Learning Rate 감소가 정확도에 부정적 영향({lr_diff:.2f}%)을 미쳤습니다.")
    
    # LoRA Rank 영향 분석 (exp-002 vs exp-003)
    if 'exp-002' in accuracies and 'exp-003' in accuracies:
        rank_diff = accuracies['exp-003'] - accuracies['exp-002']
        if rank_diff > 0:
            insights.append(f"LoRA Rank 증가(16 → 32)로 정확도가 {rank_diff:.2f}% 향상되었습니다.")
        else:
            insights.append(f"LoRA Rank 증가가 오히려 정확도를 {abs(rank_diff):.2f}% 감소시켰습니다.")

print("\n분석 인사이트:")
for i, insight in enumerate(insights, 1):
    print(f"  {i}. {insight}")

## 6. 최종 보고서 생성

In [ ]:
# 최종 보고서 생성
report = {
    "report_generated_at": datetime.now().isoformat(),
    "project": "농업 도메인 Tool Calling sLLM Fine-tuning",
    "base_model": "Qwen2.5-7B-Instruct",
    "experiments": [],
    "best_experiment": None,
    "insights": insights,
}

for exp in EXPERIMENTS:
    if exp not in configs:
        continue
    
    exp_report = {
        "name": exp,
        "description": configs[exp]['experiment']['description'],
        "config": {
            "learning_rate": configs[exp]['training']['learning_rate'],
            "batch_size": configs[exp]['training']['per_device_train_batch_size'],
            "epochs": configs[exp]['training']['num_train_epochs'],
            "lora_rank": configs[exp]['lora']['r'],
            "lora_alpha": configs[exp]['lora']['lora_alpha'],
        },
        "results": {}
    }
    
    # 학습 결과 추가
    if exp in training_logs:
        train_losses = [x['loss'] for x in training_logs[exp] if 'loss' in x and 'eval_loss' not in x]
        eval_losses = [x['eval_loss'] for x in training_logs[exp] if 'eval_loss' in x]
        
        exp_report['results']['training'] = {
            "final_train_loss": train_losses[-1] if train_losses else None,
            "final_eval_loss": eval_losses[-1] if eval_losses else None,
            "min_eval_loss": min(eval_losses) if eval_losses else None,
        }
    
    # 평가 결과 추가
    if exp in evaluation_results:
        exp_report['results']['evaluation'] = evaluation_results[exp]
    
    report['experiments'].append(exp_report)

# 최고 성능 실험 기록
if eval_available_exps:
    report['best_experiment'] = {
        "name": best_exp,
        "accuracy": best_accuracy
    }

print("최종 보고서 생성 완료")

In [ ]:
# 보고서 저장
report_path = '../results/final_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"최종 보고서 저장: {report_path}")

In [ ]:
# 결과 요약 출력
print("\n" + "=" * 70)
print("농업 도메인 Tool Calling sLLM Fine-tuning 실험 최종 보고서")
print("=" * 70)

print(f"\n생성 일시: {report['report_generated_at']}")
print(f"Base Model: {report['base_model']}")

print("\n" + "-" * 70)
print("실험 결과 요약")
print("-" * 70)

for exp_report in report['experiments']:
    print(f"\n[{exp_report['name']}] {exp_report['description']}")
    print(f"  설정: LR={exp_report['config']['learning_rate']}, Rank={exp_report['config']['lora_rank']}, Epochs={exp_report['config']['epochs']}")
    
    if 'training' in exp_report['results']:
        train_res = exp_report['results']['training']
        print(f"  학습: Train Loss={train_res['final_train_loss']:.4f}, Eval Loss={train_res['final_eval_loss']:.4f}" 
              if train_res['final_train_loss'] else "  학습: 결과 없음")
    
    if 'evaluation' in exp_report['results']:
        eval_res = exp_report['results']['evaluation']
        print(f"  평가: 정확도={eval_res['overall_accuracy']:.2f}%")

if report['best_experiment']:
    print("\n" + "-" * 70)
    print(f"최고 성능: {report['best_experiment']['name']} ({report['best_experiment']['accuracy']:.2f}%)")

if report['insights']:
    print("\n" + "-" * 70)
    print("주요 인사이트:")
    for insight in report['insights']:
        print(f"  - {insight}")

print("\n" + "=" * 70)
print("분석 완료!")
print("=" * 70)

In [ ]:
# 저장된 파일 목록
print("\n저장된 결과 파일:")
print(f"  - results/final_report.json")
print(f"  - results/loss_comparison.png")
print(f"  - results/tool_accuracy_comparison.png")
print(f"  - results/complexity_accuracy_heatmap.png")
print(f"  - results/error_distribution_comparison.png")